# Bulk RNA-seq Time-Course Analysis Template

本 notebook 用于处理时间梯度 RNA-seq 数据，包括分组均值聚合、Mfuzz 软聚类、趋势可视化、热图，以及各时间点的差异表达分析。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
EXPR_FILE <- "./1-DEG/vsd_matrix.csv"     # genes x samples; exported by RNAseq_General
META_FILE <- "./1-DEG/colData.csv"         # must contain sample, condition, and time columns
GENE_COLUMN <- NULL                         # NULL = use rownames/first column
SAMPLE_COLUMN <- "sample"
TIME_COLUMN <- "time"                      # numeric or factor time column
GROUP_COLUMN <- "condition"               # optional treatment/group column
TIME_LEVELS <- NULL                         # e.g. c("Day0", "Day7", "Day14", "Day21")

# Time-course clustering
RUN_MFUZZ <- TRUE
MFUZZ_N_CLUSTERS <- 5
MFUZZ_MIN_ACORE <- 0.7
MFUZZ_SEED <- 2025

# Pairwise DEG: each time point vs baseline
RUN_TIMEPOINT_DEG <- TRUE
BASELINE_TIME <- NULL                        # NULL = earliest TIME_LEVELS; or specify e.g. "Day0"
DEG_PADJ_CUTOFF <- 0.05
DEG_LFC_CUTOFF <- 0.5

# Output
OUTDIR <- "RNAseq_TimeCourse_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("tidyverse"))
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("Mfuzz", "ComplexHeatmap", "circlize", "clusterProfiler", "org.Hs.eg.db"))

suppressPackageStartupMessages({
  library(tidyverse)
  library(ComplexHeatmap)
  library(circlize)
  library(Mfuzz)
  library(clusterProfiler)
  library(org.Hs.eg.db)
})

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
source(file.path(LIB_DIR, "deg_utils.R"))
source(file.path(LIB_DIR, "enrichment_utils.R"))
source(file.path(LIB_DIR, "timecourse_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")


## 3. Load Expression and Metadata

In [ ]:
expr_raw <- read.csv(EXPR_FILE, check.names = FALSE)
if (!is.null(GENE_COLUMN) && GENE_COLUMN %in% colnames(expr_raw)) {
  genes <- expr_raw[[GENE_COLUMN]]
  expr <- as.matrix(expr_raw[, setdiff(colnames(expr_raw), GENE_COLUMN), drop = FALSE])
  rownames(expr) <- genes
} else if (!is.numeric(expr_raw[[1]])) {
  genes <- expr_raw[[1]]
  expr <- as.matrix(expr_raw[, -1, drop = FALSE])
  rownames(expr) <- genes
} else {
  expr <- as.matrix(expr_raw)
}
mode(expr) <- "numeric"
expr <- expr[!duplicated(rownames(expr)) & !is.na(rownames(expr)) & rownames(expr) != "", , drop = FALSE]

meta <- read.csv(META_FILE, check.names = FALSE)
if (is.null(TIME_LEVELS)) TIME_LEVELS <- unique(as.character(meta[[TIME_COLUMN]]))
meta[[TIME_COLUMN]] <- factor(meta[[TIME_COLUMN]], levels = TIME_LEVELS)
common_samples <- intersect(colnames(expr), meta[[SAMPLE_COLUMN]])
expr <- expr[, common_samples, drop = FALSE]
meta <- meta[match(common_samples, meta[[SAMPLE_COLUMN]]), ]
stopifnot(all(colnames(expr) == meta[[SAMPLE_COLUMN]]))

cat("Expression:", nrow(expr), "genes x", ncol(expr), "samples\n")
print(table(meta[[TIME_COLUMN]], useNA = "ifany"))


## 4. Aggregate Expression by Time Point

In [ ]:
expr_mean <- aggregate_expr_by_group(expr, meta[[TIME_COLUMN]])
expr_mean <- expr_mean[, TIME_LEVELS, drop = FALSE]  # ensure order
write.csv(expr_mean, file.path(OUTDIR, "mean_expression_by_time.csv"))
cat("Aggregated expression:", nrow(expr_mean), "genes x", ncol(expr_mean), "time points\n")


## 5. Mfuzz Time-Course Soft Clustering

In [ ]:
if (RUN_MFUZZ) {
  eset <- prepare_mfuzz_eset(expr_mean)
  mfuzz_result <- run_mfuzz(eset, n_clusters = MFUZZ_N_CLUSTERS, seed = MFUZZ_SEED)
  cluster_df <- extract_mfuzz_clusters(mfuzz_result, eset = eset, min_acore = MFUZZ_MIN_ACORE)
  write_mfuzz_cluster_table(cluster_df, file.path(OUTDIR, "mfuzz_clusters.csv"))

  cat("Mfuzz cluster sizes:\n")
  print(summarize_mfuzz_clusters(cluster_df))

  # Trend plots
  plot_mfuzz_trends_pdf(
    eset, mfuzz_result,
    filename = file.path(OUTDIR, "mfuzz_trends.pdf"),
    time_labels = TIME_LEVELS,
    width = 14, height = 10
  )

  # Heatmap of core genes by cluster
  core_df <- cluster_df[cluster_df$core_gene, ]
  if (nrow(core_df) >= 2) {
    group_colors <- make_group_colors(TIME_LEVELS)
    plot_timecourse_heatmap_pdf(
      expr, core_df,
      group_vec = meta[[TIME_COLUMN]],
      group_levels = TIME_LEVELS,
      group_colors = group_colors,
      filename = file.path(OUTDIR, "mfuzz_core_heatmap.pdf"),
      width = 9, height = 12
    )
  }

  # ORA per cluster
  universe <- map_symbols_to_entrez(rownames(expr), org.Hs.eg.db)$ENTREZID
  ora_results <- run_mfuzz_cluster_ora(cluster_df, org_db = org.Hs.eg.db, universe = universe)
  for (cl_name in names(ora_results)) {
    prefix <- file.path(OUTDIR, paste0("GO_ORA_", cl_name))
    write.csv(as.data.frame(ora_results[[cl_name]]), paste0(prefix, ".csv"), row.names = FALSE)
    plot_enrich_suite_pdf(ora_results[[cl_name]], prefix, cl_name)
  }
}


## 6. Time-Point vs Baseline DEG (Optional)

In [ ]:
# This section requires raw counts and is meant to be adapted.
# For now we document the recommended approach:
#   1. Use RNAseq_General.ipynb with subsetted samples at each time point.
#   2. Or loop over time points and run DESeq2 with design ~ condition + time.
cat("Time-point DEG section: integrate with RNAseq_General or RNAseq_limma_voom as needed.\n")


## 7. Save Session

In [ ]:
save.image(file = file.path(OUTDIR, "timecourse_workspace.Rdata"))
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
cat("Analysis complete. Outputs saved to", OUTDIR, "\n")
